In [ ]:
"""General-purpose curation pipeline for QSAR datasets.

Pipeline order (each step depends on the previous one):
    prepare -> ChEMBL standardize -> ChEMBL parent -> largest fragment
    -> neutralize -> species filter -> canonical tautomer -> InChI
    -> size filter -> duplicate removal

Three ordering constraints are load-bearing:

1. `get_parent_molblock` does not strip every counter-ion. ChEMBL's fragment
   parent uses skip_if_all_match=True, so when *every* fragment is a known
   salt the record is left untouched (sodium acetate, valerate and benzoate
   are all on ChEMBL's salt list). That is why the largest-fragment step
   still runs afterwards.

2. `get_parent_molblock` knows which fragments are counter-ions but the
   largest-fragment chooser only knows sizes, so the two cover different
   failure modes: the ChEMBL parent saves records whose counter-ion is bigger
   than the compound (e.g. a pamoate salt), while the largest-fragment step
   saves records the ChEMBL rule deliberately skipped.

3. The Uncharger balances charge over the whole molecule, so it will NOT
   protonate an anion while a non-neutralizable cation (e.g. a quaternary
   ammonium) sits in another fragment. It must run AFTER the largest-fragment
   step, which cuts without neutralizing. The species filter in turn runs
   AFTER the Uncharger, so counter-ions are matched in their neutral form
   (sulfate as OS(=O)(=O)O, not as [O-]S(=O)(=O)[O-]).

The log separates rows actually dropped from structures that were merely
rewritten, and asserts that initial - removed == final. Every removed row
carries its `original_index`, so any drop can be traced to the input file.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import rdkit
from rdkit import Chem, RDLogger
from rdkit.Chem import inchi as rd_inchi
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
import chembl_structure_pipeline as csp
from chembl_structure_pipeline import standardizer

RDLogger.DisableLog('rdApp.*')  # errors are captured and logged per stage instead


# =============================== Structure ===============================
def _validate_elements(symbols: frozenset[str]) -> frozenset[str]:
    """Fail fast on typos: an unrecognized symbol would silently reject
    every molecule containing that element."""
    table = Chem.GetPeriodicTable()
    unknown = []
    for symbol in symbols:
        try:
            table.GetAtomicNumber(symbol)
        except RuntimeError:
            unknown.append(symbol)
    if unknown:
        raise ValueError(f'unknown element symbol(s): {sorted(unknown)}')
    return symbols


# Elements the downstream descriptors/fingerprints are expected to handle.
# Anything else is rejected: this is what removes organometallics, metal
# complexes and inorganic residues, without guessing from composition.
# Drop 'B', 'Si' or 'Se' if the descriptor set does not cover them.
ALLOWED_ELEMENTS = _validate_elements(frozenset(
    {'H', 'B', 'C', 'N', 'O', 'F', 'Si', 'P', 'S', 'Cl', 'Se', 'Br', 'I'}))

# Merge enantiomers/diastereomers. True is the usual choice for 2D
# descriptors and fingerprints without chirality. Set to False when the
# descriptors encode stereochemistry and the data supports the distinction.
REMOVE_STEREOCHEMISTRY = True

# False -> aromatic canonical SMILES (recommended: stable identifier, since
# different aromaticity perceptions can yield different Kekule forms for the
# same molecule). Set to True only if a downstream tool requires it.
KEKULIZE_OUTPUT = False

# Size bounds; set any to None to disable. With no carbon requirement in the
# species filter, these are the only guard against tiny species (water, H2S).
# MIN_HEAVY_ATOMS = 3 suits drug-like space; lower it for fragments,
# solvents or gases.
MIN_HEAVY_ATOMS = 3
MIN_MOLECULAR_MASS = None
MAX_MOLECULAR_MASS = 1000

# Duplicate grouping key:
#   'inchikey' -> full InChIKey (stereo- and isotope-aware)
#   'skeleton' -> first InChIKey block, which ignores stereo, isotopes and
#                 protonation. Use with REMOVE_STEREOCHEMISTRY = False when
#                 stereoisomers must stay in the file but be collapsed for
#                 2D modelling.
DUPLICATE_KEY = 'inchikey'

# ================================= Target ================================
# How the endpoint relates to a log scale:
#   'none'  -> already log-like or linear (pIC50, logP, logS, pKa, dG,
#              percentage). Judged and averaged as given; negatives allowed.
#   'log10' -> concentration-like and strictly positive (IC50, Ki, EC50 in
#              nM/uM). Judged in log10, collapsed to the geometric mean.
#   'ln'    -> same, natural log.
TARGET_TRANSFORM = 'log10'

# Max standard deviation tolerated among replicates of the same structure,
# in the unit set by TARGET_TRANSFORM. With 'log10', 0.2 is a ~1.58x spread.
VARIANCE_THRESHOLD = 0.2

# Population std (ddof=0). ddof=1 inflates the spread by sqrt(2) for n=2 and
# pushes concordant pairs into the discordant pile.
STD_DDOF = 0

_uncharger = rdMolStandardize.Uncharger()
_largest_fragment_chooser = rdMolStandardize.LargestFragmentChooser()
_tautomer_enumerator = rdMolStandardize.TautomerEnumerator()

# Metals by atomic number: alkali, alkaline earth, transition,
# post-transition, lanthanides and actinides.
_METAL_ATOMIC_NUMBERS = frozenset(
    list(range(3, 5)) + list(range(11, 14)) + list(range(19, 32))
    + list(range(37, 51)) + list(range(55, 85)) + list(range(87, 104))
) - frozenset({5, 6, 7, 8, 9, 10, 14, 15, 16, 17, 18, 33, 34, 35, 36, 52, 53,
               54, 85})

# Counter-ions written in their neutral form, i.e. as they look after the
# Uncharger stage. Canonicalized at import, so input notation is irrelevant.
_COUNTER_ION_INPUTS = [
    'Cl', 'Br', 'I', 'F',                     # hydrohalic acids
    'OS(=O)(=O)O', 'OS(=O)O',                 # sulfate, sulfite
    'O[N+](=O)[O-]', 'ON=O',                  # nitrate, nitrite
    'OP(=O)(O)O', 'OP(O)O',                   # phosphate, phosphite
    'OC(=O)O',                                # carbonate: has carbon, still a residue
    'OCl(=O)(=O)=O', 'OCl=O', 'OCl',          # perchlorate, chlorite, hypochlorite
    'O[Si](O)(O)O',                           # silicate
    'OB(O)O',                                 # borate
    'O=S(=O)=O',                              # sulfur trioxide
]


def _canonical_set(smiles_list: list[str]) -> frozenset[str]:
    return frozenset(
        Chem.MolToSmiles(m) for m in map(Chem.MolFromSmiles, smiles_list)
        if m is not None)


_COUNTER_IONS = _canonical_set(_COUNTER_ION_INPUTS)


def _to_smiles(mol: Chem.Mol | None) -> str | None:
    """Canonical SMILES. Kekule output is optional and always preceded by Kekulize()."""
    if mol is None:
        return None
    if not KEKULIZE_OUTPUT:
        return Chem.MolToSmiles(mol)
    mol = Chem.Mol(mol)
    Chem.Kekulize(mol, clearAromaticFlags=True)
    return Chem.MolToSmiles(mol, kekuleSmiles=True)


def _canonical_or_self(smile: str) -> str:
    """Canonicalize for comparison, so plain recanonicalization isn't a change."""
    mol = Chem.MolFromSmiles(smile) if isinstance(smile, str) else None
    return Chem.MolToSmiles(mol) if mol is not None else str(smile)


# ================================== Log ==================================
class CurationLog:
    """Keeps REMOVED ROWS separate from MODIFIED STRUCTURES (row kept)."""

    def __init__(self) -> None:
        self.removed: dict[str, int] = {}
        self.modified: dict[str, int] = {}
        self.audit: list[pd.DataFrame] = []

    def add_removed(self, label: str, n: int) -> None:
        if n:
            self.removed[label] = self.removed.get(label, 0) + int(n)

    def add_modified(self, label: str, n: int) -> None:
        if n:
            self.modified[label] = self.modified.get(label, 0) + int(n)

    def add_audit(self, stage: str, index, before: pd.Series, after: pd.Series,
                  raw_input: bool = False) -> None:
        if raw_input:
            before = before.apply(_canonical_or_self)
        changed = (before != after).values
        if not changed.any():
            return
        self.audit.append(pd.DataFrame({
            'original_index': np.asarray(index)[changed],
            'stage': stage,
            'smiles_before': before.values[changed],
            'smiles_after': after.values[changed],
        }))
        self.add_modified(f'Structure changed in "{stage}"', int(changed.sum()))


def _apply_stage(df, log, stage, func, raw_input=False):
    """Apply func(smiles) -> (new_smiles | None, reason | None).

    Returns (kept df, removed df). No row is counted as removed when the
    structure was merely modified.
    """
    before = df['final_smiles'].copy()
    results = before.apply(func)
    new_smiles = results.apply(lambda r: r[0])
    reasons = results.apply(lambda r: r[1])

    failed = new_smiles.isna()
    removed_df = df[failed].copy()
    if not removed_df.empty:
        removed_df['removal_reason'] = f'{stage}: ' + reasons[failed]
        for reason, n in reasons[failed].value_counts().items():
            log.add_removed(f'{stage}: {reason}', n)

    kept = df[~failed].copy()
    kept['final_smiles'] = new_smiles[~failed].values
    log.add_audit(stage, kept['original_index'], before[~failed],
                  new_smiles[~failed], raw_input=raw_input)
    return kept.reset_index(drop=True), removed_df


# ================================ Stages =================================
def prepare_structure(smile: str) -> tuple[str | None, str | None]:
    """Validate the SMILES and strip stereochemistry through the RDKit API."""
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
        return None, 'invalid SMILES'
    if REMOVE_STEREOCHEMISTRY:
        Chem.RemoveStereochemistry(mol)
    return _to_smiles(mol), None


def standardize_and_get_parent(smile: str) -> tuple[str | None, str | None]:
    """standardize_molblock + get_parent_molblock.

    The parent step strips known salts, solvates and hydrates, clears isotope
    labels, and neutralizes what it removes - none of which the
    largest-fragment chooser does.
    """
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
        return None, 'invalid SMILES'
    try:
        std_block = standardizer.standardize_molblock(Chem.MolToMolBlock(mol))
        parent_block, exclude = standardizer.get_parent_molblock(std_block)
        if exclude:
            return None, 'flagged as exclude by ChEMBL'
        parent = Chem.MolFromMolBlock(parent_block)
        if parent is None or parent.GetNumAtoms() == 0:
            return None, 'empty or unparseable parent'
        return _to_smiles(parent), None
    except Exception as exc:
        return None, f'standardization error ({type(exc).__name__})'


def keep_largest_fragment(smile: str) -> tuple[str | None, str | None]:
    """Only acts on SMILES that still hold more than one fragment."""
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
        return None, 'invalid SMILES'
    if len(Chem.GetMolFrags(mol)) <= 1:
        return _to_smiles(mol), None
    chosen = _largest_fragment_chooser.choose(mol)
    if chosen is None:
        return None, 'could not pick the largest fragment'
    return _to_smiles(chosen), None


def neutralize(smile: str) -> tuple[str | None, str | None]:
    """Safety net: neutralize charges the ChEMBL parent left behind.

    Only adds or removes hydrogens, so permanent charges survive: a
    quaternary ammonium keeps its +1, and nitro groups and N-oxides are
    untouched because their charges sit on adjacent atoms.
    """
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
        return None, 'invalid SMILES'
    try:
        return _to_smiles(_uncharger.uncharge(mol)), None
    except Exception as exc:
        return None, f'neutralization error ({type(exc).__name__})'


def canonicalize_tautomer(smile: str) -> tuple[str | None, str | None]:
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
        return None, 'invalid SMILES'
    try:
        return _to_smiles(_tautomer_enumerator.Canonicalize(mol)), None
    except Exception as exc:
        return None, f'tautomer canonicalization error ({type(exc).__name__})'


def classify_species(smile: str) -> tuple[str, str]:
    """Decide whether what is left is a modellable compound.

    Returns (verdict, detail) with verdict in {'compound', 'reject'}.
    """
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
        return 'reject', 'invalid SMILES'
    if mol.GetNumAtoms() == 0:
        return 'reject', 'empty molecule'

    heavy = [a for a in mol.GetAtoms() if a.GetAtomicNum() > 1]
    if heavy and all(a.GetAtomicNum() in _METAL_ATOMIC_NUMBERS for a in heavy):
        return 'reject', 'metal-only residue'

    if Chem.MolToSmiles(mol) in _COUNTER_IONS:
        return 'reject', 'listed inorganic counter-ion'

    offending = sorted({a.GetSymbol() for a in mol.GetAtoms()
                        if a.GetSymbol() not in ALLOWED_ELEMENTS})
    if offending:
        return 'reject', f"element not allowed: {', '.join(offending)}"

    return 'compound', ''


def species_filter_stage(df: pd.DataFrame, log: CurationLog):
    """Reject desalting residues, organometallics and out-of-scope elements.

    Runs AFTER neutralization so counter-ions are matched in neutral form.
    """
    verdicts = df['final_smiles'].apply(classify_species)
    df = df.copy()
    df['_verdict'] = verdicts.apply(lambda v: v[0])
    df['_detail'] = verdicts.apply(lambda v: v[1])

    rejected = df['_verdict'] == 'reject'
    removed_df = df[rejected].copy()
    removed_df['removal_reason'] = 'Species filter: ' + removed_df['_detail']
    for reason, n in df.loc[rejected, '_detail'].value_counts().items():
        log.add_removed(f'Species filter: {reason}', n)

    cols = ['_verdict', '_detail']
    return (df[~rejected].drop(columns=cols).reset_index(drop=True),
            removed_df.drop(columns=cols))


def inchi_stage(df: pd.DataFrame, log: CurationLog):
    """MolToInchi returns an empty string on failure - isna() would miss that."""
    def _inchi(smile):
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            return None
        value = rd_inchi.MolToInchi(mol)
        return value if value else None

    df = df.copy()
    df['InChI'] = df['final_smiles'].apply(_inchi)
    df['InChIKey'] = df['InChI'].apply(
        lambda s: rd_inchi.InchiToInchiKey(s) if s else None)
    failed = df['InChI'].isna() | df['InChIKey'].isna()
    removed_df = df[failed].copy()
    removed_df['removal_reason'] = 'InChI: calculation failed'
    log.add_removed('InChI: calculation failed', int(failed.sum()))

    kept = df[~failed].reset_index(drop=True)
    kept['dup_key'] = (kept['InChIKey'].str[:14] if DUPLICATE_KEY == 'skeleton'
                       else kept['InChIKey'])
    return kept, removed_df


def size_filter_stage(df: pd.DataFrame, log: CurationLog):
    """Heavy-atom count and molecular-mass bounds."""
    def _props(smile):
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            return pd.Series([np.nan, np.nan])
        return pd.Series([float(mol.GetNumAtoms()),
                          rdMolDescriptors.CalcExactMolWt(mol)])

    df = df.copy()
    df[['heavy_atoms', 'molecular_mass']] = df['final_smiles'].apply(_props)

    reasons = pd.Series('', index=df.index)
    invalid = df['molecular_mass'].isna()
    reasons[invalid] = 'could not compute properties'
    if MIN_HEAVY_ATOMS is not None:
        hit = ~invalid & (reasons == '') & (df['heavy_atoms'] < MIN_HEAVY_ATOMS)
        reasons[hit] = f'fewer than {MIN_HEAVY_ATOMS} heavy atoms'
    if MIN_MOLECULAR_MASS is not None:
        hit = ~invalid & (reasons == '') & (df['molecular_mass'] < MIN_MOLECULAR_MASS)
        reasons[hit] = f'molecular mass < {MIN_MOLECULAR_MASS}'
    if MAX_MOLECULAR_MASS is not None:
        hit = ~invalid & (reasons == '') & (df['molecular_mass'] > MAX_MOLECULAR_MASS)
        reasons[hit] = f'molecular mass > {MAX_MOLECULAR_MASS}'

    failed = reasons != ''
    removed_df = df[failed].copy()
    removed_df['removal_reason'] = 'Size filter: ' + reasons[failed]
    for reason, n in reasons[failed].value_counts().items():
        log.add_removed(f'Size filter: {reason}', n)
    return df[~failed].reset_index(drop=True), removed_df


# ============================== Duplicates ===============================
def _finish_dedup(df, keep, conc, disc, log):
    empty = pd.DataFrame(columns=df.columns)
    final = pd.concat(keep, ignore_index=True) if keep else empty.copy()
    rem_c = pd.concat(conc, ignore_index=True) if conc else empty.copy()
    rem_d = pd.concat(disc, ignore_index=True) if disc else empty.copy()
    log.add_removed('Concordant duplicates (1 row kept)', len(rem_c))
    log.add_removed('Discordant duplicates (whole group discarded)', len(rem_d))
    return final, rem_c, rem_d


def remove_duplicates_classification(df, outcome_col, log):
    """Concordant: keep one row. Discordant: drop the whole group."""
    df = df.copy()
    df['n_replicates'] = 1
    keep, conc, disc = [], [], []
    for _, group in df.groupby('dup_key', sort=False):
        if len(group) == 1:
            keep.append(group)
        elif group[outcome_col].nunique() == 1:
            rep = group.iloc[[0]].copy()
            rep['n_replicates'] = len(group)
            keep.append(rep)
            conc.append(group.iloc[1:])
        else:
            disc.append(group)
    return _finish_dedup(df, keep, conc, disc, log)


def remove_duplicates_regression(df, target_col, log,
                                 threshold=VARIANCE_THRESHOLD):
    """Concordance judged in the unit set by TARGET_TRANSFORM."""
    df = df.copy()
    df[target_col] = pd.to_numeric(df[target_col], errors='coerce')

    bad = df[target_col].isna()
    removed_invalid = df[bad].copy()
    removed_invalid['removal_reason'] = f'non-numeric value in "{target_col}"'
    log.add_removed(f'Non-numeric value in "{target_col}"', int(bad.sum()))
    df = df[~bad].reset_index(drop=True)

    if TARGET_TRANSFORM in ('log10', 'ln'):
        nonpos = df[target_col] <= 0
        extra = df[nonpos].copy()
        extra['removal_reason'] = (
            f'non-positive value in "{target_col}", incompatible with '
            f'TARGET_TRANSFORM={TARGET_TRANSFORM}')
        removed_invalid = pd.concat([removed_invalid, extra], ignore_index=True)
        log.add_removed(f'Non-positive value in "{target_col}"', int(nonpos.sum()))
        df = df[~nonpos].reset_index(drop=True)
        log_fn = np.log10 if TARGET_TRANSFORM == 'log10' else np.log
        df['dedup_scale'] = log_fn(df[target_col])
    elif TARGET_TRANSFORM == 'none':
        df['dedup_scale'] = df[target_col]
    else:
        raise ValueError("TARGET_TRANSFORM must be 'none', 'log10' or 'ln'")

    df['n_replicates'] = 1
    keep, conc, disc = [], [], []
    for _, group in df.groupby('dup_key', sort=False):
        if len(group) == 1:
            keep.append(group)
            continue
        if group['dedup_scale'].std(ddof=STD_DDOF) <= threshold:
            rep = group.iloc[[0]].copy()
            mean_scale = group['dedup_scale'].mean()
            if TARGET_TRANSFORM == 'log10':
                rep[target_col] = 10 ** mean_scale
            elif TARGET_TRANSFORM == 'ln':
                rep[target_col] = np.exp(mean_scale)
            else:
                rep[target_col] = mean_scale
            rep['dedup_scale'] = mean_scale
            rep['n_replicates'] = len(group)
            keep.append(rep)
            conc.append(group.iloc[1:])
        else:
            disc.append(group)

    out, rem_c, rem_d = _finish_dedup(df, keep, conc, disc, log)
    return out, rem_c, rem_d, removed_invalid


# ================================ Output =================================
def _save(df, path, label):
    if df is not None and not df.empty:
        df.to_csv(path, index=False)
        print(f"  -> {label}: {len(df)} rows in '{path}'")


def write_log(savepath: Path, log: CurationLog, initial: int, final: int,
              settings: dict) -> None:
    total_removed = sum(log.removed.values())
    lines = ['QSAR curation log', '=' * 64,
             f'RDKit {rdkit.__version__} | '
             f'chembl_structure_pipeline {csp.__version__}',
             '', 'SETTINGS', '-' * 64]
    lines += [f'{k}: {v}' for k, v in settings.items()]
    lines += ['', f'Initial rows: {initial}', '', 'REMOVED ROWS', '-' * 64]
    lines += [f'{k}: {v}' for k, v in log.removed.items()] or ['(none)']
    lines += ['-' * 64,
              f'Total rows removed: {total_removed}',
              '',
              f'Final rows: {final}',
              f'Sanity check (initial - removed == final): '
              f'{initial - total_removed == final}',
              '',
              'MODIFIED STRUCTURES (row kept, not counted as a removal)',
              '-' * 64]
    lines += [f'{k}: {v}' for k, v in log.modified.items()] or ['(none)']
    (savepath / 'curation_log.txt').write_text('\n'.join(lines) + '\n')
    print('\n'.join(lines))


def curate_dataset(df, smiles_col, outcome_col, task_type='classification',
                   savepath='curated_data'):
    """Curate a QSAR dataset and write the results to `savepath`.

    Writes curated_dataset.csv, removed_rows.csv (every drop except
    duplicates, with a `removal_reason`), removed_concordant_duplicates.csv,
    removed_discordant_duplicates.csv, modified_structures.csv (audit trail,
    not removals) and curation_log.txt. Empty tables are not written.
    """
    out_dir = Path(savepath)
    out_dir.mkdir(parents=True, exist_ok=True)
    log = CurationLog()
    initial = len(df)
    removed_parts = []

    settings = {
        'task_type': task_type,
        'allowed_elements': ' '.join(sorted(ALLOWED_ELEMENTS)),
        'remove_stereochemistry': REMOVE_STEREOCHEMISTRY,
        'kekulize_output': KEKULIZE_OUTPUT,
        'min_heavy_atoms': MIN_HEAVY_ATOMS,
        'molecular_mass_range': f'{MIN_MOLECULAR_MASS} - {MAX_MOLECULAR_MASS}',
        'duplicate_key': DUPLICATE_KEY,
        'target_transform': TARGET_TRANSFORM,
        'variance_threshold': VARIANCE_THRESHOLD,
        'std_ddof': STD_DDOF,
    }

    df = df.copy().reset_index(drop=True)
    df.insert(0, 'original_index', df.index)

    missing = df[smiles_col].isna() | df[outcome_col].isna()
    log.add_removed('Missing SMILES or outcome', int(missing.sum()))
    removed_parts.append(
        df[missing].assign(removal_reason='missing SMILES/outcome'))
    df = df[~missing].reset_index(drop=True)
    df['final_smiles'] = df[smiles_col]

    for i, (name, func) in enumerate([
        ('Preparation', prepare_structure),
        ('ChEMBL standardization + parent', standardize_and_get_parent),
        ('Largest fragment', keep_largest_fragment),
        ('Neutralization', neutralize),
    ]):
        df, removed = _apply_stage(df, log, name, func, raw_input=(i == 0))
        removed_parts.append(removed)
        print(f'{name}: {len(df)} rows remaining')

    df, removed = species_filter_stage(df, log)
    removed_parts.append(removed)
    print(f'Species filter: {len(df)} rows remaining')

    df, removed = _apply_stage(df, log, 'Canonical tautomer',
                               canonicalize_tautomer)
    removed_parts.append(removed)
    print(f'Canonical tautomer: {len(df)} rows remaining')

    df, removed = inchi_stage(df, log)
    removed_parts.append(removed)
    print(f'InChI: {len(df)} rows remaining')

    df, removed = size_filter_stage(df, log)
    removed_parts.append(removed)
    print(f'Size filter: {len(df)} rows remaining')

    if task_type == 'classification':
        df, rem_c, rem_d = remove_duplicates_classification(df, outcome_col, log)
    elif task_type == 'regression':
        df, rem_c, rem_d, rem_inv = remove_duplicates_regression(
            df, outcome_col, log)
        removed_parts.append(rem_inv)
    else:
        raise ValueError("task_type must be 'classification' or 'regression'")
    print(f'Duplicate removal: {len(df)} rows remaining')

    final = len(df)
    _save(rem_c, out_dir / 'removed_concordant_duplicates.csv',
          'concordant duplicates')
    _save(rem_d, out_dir / 'removed_discordant_duplicates.csv',
          'discordant duplicates')
    nonempty = [p for p in removed_parts if not p.empty]
    _save(pd.concat(nonempty, ignore_index=True) if nonempty else pd.DataFrame(),
          out_dir / 'removed_rows.csv', 'removed rows')
    _save(pd.concat(log.audit, ignore_index=True) if log.audit else pd.DataFrame(),
          out_dir / 'modified_structures.csv', 'modified structures')

    df = df.drop(columns=[smiles_col], errors='ignore')
    df.to_csv(out_dir / 'curated_dataset.csv', index=False)
    write_log(out_dir, log, initial, final, settings)
    return df, log

In [ ]:
df = pd.read_csv('dataset.csv')

# IC50/Ki in nM -> set TARGET_TRANSFORM = 'log10' before calling.
# pIC50, logP, logS, pKa -> leave TARGET_TRANSFORM = 'none'.

curated, log = curate_dataset(
    df,
    smiles_col='SMILES',
    outcome_col='pIC50',
    task_type='regression',
    savepath='./curated',
)